[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/047.%20The%20Demand%20Forecasting%20-%20Moving%20Average%20%26%20Weighted%20MA%20Problem/P47-Tier-4_executed.ipynb)

---


# 47. The Demand Forecasting Problem
## Tier 4: LSTM-Augmented Weighted Moving Average

### Goal and scope

This notebook demonstrates a one-step-ahead demand forecast that combines a Long Short-Term Memory (LSTM) model with an enhanced weighted moving average (WMA). The LSTM is implemented with NumPy and trained by backpropagation through time using the Adam optimizer. The final forecast is a weighted combination of the two component forecasts.

The input is a sequence of demand observations. The output is the next-period forecast, reported in demand units. The example begins with 12 observed weekly demands and extends them to 30 periods using a clearly identified synthetic-data rule so that the training procedure can be demonstrated. Synthetic periods are not additional observations from a real business.

### Model assumptions

- Demand is represented as a single numerical time series with equally spaced periods.
- Each training example uses the preceding six periods to predict the next period.
- The LSTM has one recurrent layer and a linear output layer. The network is trained on the generated example; no claim is made that this small example establishes real-world forecast superiority.
- The WMA assigns greater weight to more recent observations and uses at most the latest four periods.
- The hybrid weight is specified explicitly for demonstration. It is not estimated or proven optimal by this notebook.
- The agreement-based confidence label is a simple diagnostic, not a calibrated probability or prediction interval.

Real forecasting applications also require decisions about seasonality, structural changes, forecast horizons, data quality, and evaluation on genuinely unseen observations. Those topics are outside this example's scope.

### Data

The observed demand series is `[450, 380, 520, 610, 485, 390, 475, 550, 425, 480, 515, 460]` units. Additional periods are generated from the observed series' mean and standard deviation, a small linear trend, an eight-period sinusoidal component, and random noise. A fixed random seed makes the demonstration reproducible.

### Solution sequence

1. Scale the series using the training portion only, then create input-target sequences.
2. Process each sequence with the LSTM gates and update the network parameters by backpropagation through time.
3. Monitor a later chronological validation segment and stop when its loss no longer improves.
4. Produce LSTM and WMA forecasts and combine them with the stated hybrid weight.
5. Inspect the forecast components, validation metrics, and visualizations without treating the synthetic example as evidence of general accuracy.

### Interpreting the results

The reported validation metrics describe performance on the held-out portion of this generated series. The final forecast is for the period immediately after the 30 generated periods. It should be interpreted as a demonstration of the method, not as a business-ready demand plan. The notebook does not estimate uncertainty intervals or optimize the ensemble weight.

## LSTM-Enhanced Hybrid Forecasting Architecture

### LSTM cell

An LSTM cell carries a hidden state and a cell state from one time step to the next. The cell state acts as a memory pathway; gates control how much information is retained, added, and exposed. This gated recurrence is useful for representing dependencies across a sequence. In this notebook, each observation is a scalar, and the final hidden state is passed to a linear layer to predict the next demand value.

For time step $t$, the cell computes:

$$
\begin{aligned}
f_t &= \sigma\!\left(W_f [h_{t-1}, x_t] + b_f\right), \\
i_t &= \sigma\!\left(W_i [h_{t-1}, x_t] + b_i\right), \\
\tilde{C}_t &= \tanh\!\left(W_C [h_{t-1}, x_t] + b_C\right), \\
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t, \\
o_t &= \sigma\!\left(W_o [h_{t-1}, x_t] + b_o\right), \\
h_t &= o_t \odot \tanh(C_t), \\
\hat{y} &= W_{\mathrm{out}} h_T + b_{\mathrm{out}}.
\end{aligned}
$$

Here, $x_t$ is the observation at step $t$; $h_t$ is the hidden state; $C_t$ is the cell state; $f_t$, $i_t$, and $o_t$ are the forget, input, and output gates; and $\tilde{C}_t$ is the candidate cell content. The symbol $\odot$ denotes elementwise multiplication, while $[h_{t-1},x_t]$ denotes concatenation. The matrices $W$ and vectors $b$ are learned parameters. The functions $\sigma(z)=1/(1+e^{-z})$ and $\tanh(z)$ are applied elementwise. In the code, `LSTMCell.forward` implements the gate equations, `SimpleLSTM.forward` processes a sequence, and `W_out` and `b_out` form the linear prediction layer.

### Training objective and recurrence through time

For a training set of $N$ sequence-target pairs, the network minimizes mean squared error:

$$
L(\theta) = \frac{1}{N}\sum_{j=1}^{N}(\hat{y}_j-y_j)^2,
$$

where $y_j$ is the observed next-period demand on the scaled axis, $\hat{y}_j$ is the model prediction, and $\theta$ denotes all gate and output parameters. Backpropagation through time applies the chain rule from the output back through the recurrent steps; Adam uses these gradients to update the parameters. The implementation computes a gradient for each training sequence and updates the parameters sequentially. Training stops early when chronological validation loss has not improved for the specified patience.

If there are $T$ time steps and hidden size $H$, a forward or backward pass for one sequence has computational cost proportional to $T H(H+1)$ for scalar inputs, up to constant factors. Training cost also scales with the number of sequences and epochs. The six-period input window bounds the recurrent computation used in this example.

### Hybrid forecast

The two component forecasts are combined as:

$$
F_{\mathrm{hybrid}} = \alpha F_{\mathrm{LSTM}} + (1-\alpha)F_{\mathrm{WMA}}, \qquad 0 \leq \alpha \leq 1.
$$

Here, $F_{\mathrm{LSTM}}$ and $F_{\mathrm{WMA}}$ are the forecasts in the original demand units, and $\alpha$ is the stated LSTM share. The code uses `lstm_weight` for $\alpha$ and `wma_weight` for $1-\alpha$. This is a fixed illustrative blend, not a weight learned from validation data.

### WMA definition

For a window of $m$ observations, the WMA is

$$
F_{\mathrm{WMA}} = \sum_{j=1}^{m} w_j d_j, \qquad
w_j = \frac{2^{-(m-j)}}{\sum_{k=1}^{m}2^{-(m-k)}},
$$

where $d_j$ is the $j$th observation in the selected window, ordered from oldest to most recent. The weights sum to one, so the forecast is a weighted average in the same units as demand. The function `generate_enhanced_wma_forecast` implements this calculation using at most the latest four observations.

In [ ]:
import warnings
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")
np.random.seed(47)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("NumPy-based LSTM forecasting components are ready.")

In [ ]:
class CustomMinMaxScaler:
    """Scale a single feature using training-data minimum and maximum."""

    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
        self.min_ = None
        self.scale_ = None
        self.data_min_ = None
        self.data_max_ = None

    def fit(self, X):
        """Learn the scaling parameters from the supplied data."""
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        if X.ndim != 2 or X.shape[0] == 0:
            raise ValueError("X must contain at least one observation.")

        self.data_min_ = X.min(axis=0)
        self.data_max_ = X.max(axis=0)
        data_range = self.data_max_ - self.data_min_
        safe_range = np.where(data_range == 0, 1.0, data_range)
        lower, upper = self.feature_range
        self.scale_ = (upper - lower) / safe_range
        self.min_ = lower - self.data_min_ * self.scale_
        return self

    def transform(self, X):
        """Apply the scaling parameters learned by fit."""
        if self.scale_ is None:
            raise ValueError("Call fit before transform.")
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return X * self.scale_ + self.min_

    def fit_transform(self, X):
        """Fit the scaler and transform the same data."""
        return self.fit(X).transform(X)

    def inverse_transform(self, X):
        """Return scaled values to the original feature units."""
        if self.scale_ is None:
            raise ValueError("Call fit before inverse_transform.")
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        return (X - self.min_) / self.scale_


def mean_absolute_error(y_true, y_pred):
    """Compute mean absolute error for aligned arrays."""
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def mean_squared_error(y_true, y_pred):
    """Compute mean squared error for aligned arrays."""
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

In [ ]:
class LSTMCell:
    """One scalar-input LSTM cell with explicit gate computations."""

    def __init__(self, input_size: int, hidden_size: int):
        self.input_size = input_size
        self.hidden_size = hidden_size
        concat_size = hidden_size + input_size
        scale = 0.1

        self.W_f = np.random.randn(hidden_size, concat_size) * scale
        self.b_f = np.zeros(hidden_size)
        self.W_i = np.random.randn(hidden_size, concat_size) * scale
        self.b_i = np.zeros(hidden_size)
        self.W_C = np.random.randn(hidden_size, concat_size) * scale
        self.b_C = np.zeros(hidden_size)
        self.W_o = np.random.randn(hidden_size, concat_size) * scale
        self.b_o = np.zeros(hidden_size)

    @staticmethod
    def sigmoid(x):
        """Compute a numerically guarded sigmoid activation."""
        x = np.clip(x, -500, 500)
        return 1.0 / (1.0 + np.exp(-x))

    def forward(self, x_t, h_prev, C_prev):
        """Advance hidden and cell states by one time step."""
        concat = np.concatenate([h_prev, x_t])
        f_t = self.sigmoid(self.W_f @ concat + self.b_f)
        i_t = self.sigmoid(self.W_i @ concat + self.b_i)
        C_tilde = np.tanh(self.W_C @ concat + self.b_C)
        C_t = f_t * C_prev + i_t * C_tilde
        o_t = self.sigmoid(self.W_o @ concat + self.b_o)
        h_t = o_t * np.tanh(C_t)
        cache = (concat, h_prev, C_prev, f_t, i_t, C_tilde, C_t, o_t)
        return h_t, C_t, cache


class SimpleLSTM:
    """Single-layer LSTM trained with BPTT and Adam for scalar forecasting."""

    def __init__(self, input_size: int, hidden_size: int, output_size: int = 1):
        if input_size != 1 or output_size != 1:
            raise ValueError("This forecasting example requires scalar input and output.")
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lstm_cell = LSTMCell(input_size, hidden_size)
        self.W_out = np.random.randn(output_size, hidden_size) * 0.1
        self.b_out = np.zeros(output_size)

    def _forward_with_cache(self, sequence):
        sequence = np.asarray(sequence, dtype=float).reshape(-1, self.input_size)
        h_t = np.zeros(self.hidden_size)
        C_t = np.zeros(self.hidden_size)
        caches = []

        for x_t in sequence:
            h_t, C_t, cache = self.lstm_cell.forward(x_t, h_t, C_t)
            caches.append(cache)

        prediction = float((self.W_out @ h_t + self.b_out)[0])
        return prediction, h_t, caches

    def forward(self, sequence):
        """Predict one scaled target from a sequence of scaled inputs."""
        prediction, _, _ = self._forward_with_cache(sequence)
        return prediction

    def _gradients(self, sequence, target):
        prediction, h_final, caches = self._forward_with_cache(sequence)
        error_gradient = 2.0 * (prediction - float(target))
        gradients = {
            "W_out": error_gradient * h_final.reshape(1, -1),
            "b_out": np.array([error_gradient]),
            "W_f": np.zeros_like(self.lstm_cell.W_f),
            "b_f": np.zeros_like(self.lstm_cell.b_f),
            "W_i": np.zeros_like(self.lstm_cell.W_i),
            "b_i": np.zeros_like(self.lstm_cell.b_i),
            "W_C": np.zeros_like(self.lstm_cell.W_C),
            "b_C": np.zeros_like(self.lstm_cell.b_C),
            "W_o": np.zeros_like(self.lstm_cell.W_o),
            "b_o": np.zeros_like(self.lstm_cell.b_o),
        }

        dh_next = error_gradient * self.W_out[0].copy()
        dC_next = np.zeros(self.hidden_size)

        for cache in reversed(caches):
            concat, _, C_prev, f_t, i_t, C_tilde, C_t, o_t = cache
            tanh_C = np.tanh(C_t)
            do = dh_next * tanh_C
            dC = dC_next + dh_next * o_t * (1.0 - tanh_C ** 2)
            df = dC * C_prev
            di = dC * C_tilde
            dC_tilde = dC * i_t
            dC_next = dC * f_t

            da_o = do * o_t * (1.0 - o_t)
            da_f = df * f_t * (1.0 - f_t)
            da_i = di * i_t * (1.0 - i_t)
            da_C = dC_tilde * (1.0 - C_tilde ** 2)

            gradients["W_o"] += np.outer(da_o, concat)
            gradients["b_o"] += da_o
            gradients["W_f"] += np.outer(da_f, concat)
            gradients["b_f"] += da_f
            gradients["W_i"] += np.outer(da_i, concat)
            gradients["b_i"] += da_i
            gradients["W_C"] += np.outer(da_C, concat)
            gradients["b_C"] += da_C

            dconcat = (
                self.lstm_cell.W_o.T @ da_o
                + self.lstm_cell.W_f.T @ da_f
                + self.lstm_cell.W_i.T @ da_i
                + self.lstm_cell.W_C.T @ da_C
            )
            dh_next = dconcat[:self.hidden_size]

        return (prediction - float(target)) ** 2, gradients

    def _parameters(self):
        return {
            "W_out": self.W_out,
            "b_out": self.b_out,
            "W_f": self.lstm_cell.W_f,
            "b_f": self.lstm_cell.b_f,
            "W_i": self.lstm_cell.W_i,
            "b_i": self.lstm_cell.b_i,
            "W_C": self.lstm_cell.W_C,
            "b_C": self.lstm_cell.b_C,
            "W_o": self.lstm_cell.W_o,
            "b_o": self.lstm_cell.b_o,
        }

    def _set_parameters(self, values):
        for name, value in values.items():
            if name in {"W_out", "b_out"}:
                setattr(self, name, value.copy())
            else:
                setattr(self.lstm_cell, name, value.copy())

    def fit(self, X, y, X_val, y_val, epochs=100, learning_rate=0.01, patience=12):
        """Train with per-sequence BPTT, Adam updates, and validation stopping."""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        X_val = np.asarray(X_val, dtype=float)
        y_val = np.asarray(y_val, dtype=float).reshape(-1)
        if len(X) == 0 or len(X_val) == 0:
            raise ValueError("Training and validation sets must both be nonempty.")
        if len(X) != len(y) or len(X_val) != len(y_val):
            raise ValueError("Each input sequence must have exactly one target.")

        parameters = self._parameters()
        first_moment = {name: np.zeros_like(value) for name, value in parameters.items()}
        second_moment = {name: np.zeros_like(value) for name, value in parameters.items()}
        beta1, beta2, epsilon = 0.9, 0.999, 1e-8
        step = 0
        best_loss = np.inf
        best_parameters = None
        epochs_without_improvement = 0
        history = {"train_loss": [], "validation_loss": []}

        for _ in range(epochs):
            for index in np.random.permutation(len(X)):
                _, gradients = self._gradients(X[index], y[index])
                step += 1
                for name, parameter in self._parameters().items():
                    gradient = gradients[name]
                    first_moment[name] = beta1 * first_moment[name] + (1 - beta1) * gradient
                    second_moment[name] = beta2 * second_moment[name] + (1 - beta2) * gradient ** 2
                    corrected_first = first_moment[name] / (1 - beta1 ** step)
                    corrected_second = second_moment[name] / (1 - beta2 ** step)
                    parameter -= learning_rate * corrected_first / (np.sqrt(corrected_second) + epsilon)

            train_predictions = np.array([self.forward(row) for row in X])
            validation_predictions = np.array([self.forward(row) for row in X_val])
            train_loss = mean_squared_error(y, train_predictions)
            validation_loss = mean_squared_error(y_val, validation_predictions)
            history["train_loss"].append(train_loss)
            history["validation_loss"].append(validation_loss)

            if validation_loss < best_loss:
                best_loss = validation_loss
                best_parameters = {
                    name: value.copy() for name, value in self._parameters().items()
                }
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= patience:
                    break

        self._set_parameters(best_parameters)
        return history

In [ ]:
class DataGenerator:
    """Extend a demand series synthetically and form supervised sequences."""

    @staticmethod
    def extend_demand_data(original_data: List[float], target_length: int = 30) -> List[float]:
        """Append synthetic values using a trend, seasonal term, and random noise."""
        if target_length < 1:
            raise ValueError("target_length must be positive.")
        if len(original_data) == 0:
            raise ValueError("original_data must contain at least one observation.")
        if len(original_data) >= target_length:
            return list(original_data[:target_length])

        extended_data = list(original_data)
        base_mean = float(np.mean(original_data))
        base_std = float(np.std(original_data))

        for i in range(target_length - len(original_data)):
            period_num = len(original_data) + i + 1
            trend_component = period_num * 2.0
            seasonal_component = 30 * np.sin(2 * np.pi * period_num / 8)
            noise = np.random.normal(0, base_std * 0.3)
            next_demand = base_mean + trend_component + seasonal_component + noise
            extended_data.append(float(np.clip(next_demand, 200, 800)))

        return extended_data

    @staticmethod
    def create_sequences(data: List[float], sequence_length: int) -> Tuple[np.ndarray, np.ndarray]:
        """Create each input window and its immediately following target."""
        if sequence_length < 1:
            raise ValueError("sequence_length must be positive.")
        X, y = [], []
        for i in range(len(data) - sequence_length):
            X.append(data[i:i + sequence_length])
            y.append(data[i + sequence_length])
        return np.asarray(X, dtype=float), np.asarray(y, dtype=float)

In [ ]:
print("=== Component checks ===")

scaler_test = CustomMinMaxScaler(feature_range=(0, 1))
test_data = np.array([100, 200, 300, 400, 500])
scaled_test_data = scaler_test.fit_transform(test_data)
restored_test_data = scaler_test.inverse_transform(scaled_test_data)
assert np.allclose(restored_test_data.ravel(), test_data)
print("Scaler round-trip check passed.")

metric_mae = mean_absolute_error([100, 200, 300], [110, 190, 310])
metric_mse = mean_squared_error([100, 200, 300], [110, 190, 310])
print(f"Metric check: MAE={metric_mae:.1f}, MSE={metric_mse:.1f}")

lstm_cell_test = LSTMCell(input_size=1, hidden_size=4)
hidden_test, cell_test, _ = lstm_cell_test.forward(
    np.array([1.0]), np.zeros(4), np.zeros(4)
)
assert hidden_test.shape == (4,) and cell_test.shape == (4,)
print("LSTM cell state-shape check passed.")

test_extended = DataGenerator.extend_demand_data([450, 380, 520, 610, 485, 390], 15)
test_X, test_y = DataGenerator.create_sequences(test_extended[:10], 3)
assert test_X.shape == (7, 3) and test_y.shape == (7,)
print("Sequence construction check passed.")

In [ ]:
original_demand = [450, 380, 520, 610, 485, 390, 475, 550, 425, 480, 515, 460]
sequence_length = 6
train_fraction = 0.8

print("=== LSTM-enhanced hybrid forecasting ===")
print(f"Observed demand periods: {len(original_demand)}")

extended_data = DataGenerator.extend_demand_data(original_demand, target_length=30)
print(f"Total periods, including synthetic data: {len(extended_data)}")

# Fit the scaler on the chronological training segment to keep later values out of scaling estimates.
train_end = int(len(extended_data) * train_fraction)
scaler = CustomMinMaxScaler(feature_range=(0, 1))
scaler.fit(np.asarray(extended_data[:train_end]).reshape(-1, 1))
scaled_data = scaler.transform(np.asarray(extended_data).reshape(-1, 1)).ravel()

X, y = DataGenerator.create_sequences(scaled_data.tolist(), sequence_length)
train_sequence_count = train_end - sequence_length
if train_sequence_count <= 0 or train_sequence_count >= len(X):
    raise ValueError("The selected data length does not provide nonempty train and validation sets.")

X_train, y_train = X[:train_sequence_count], y[:train_sequence_count]
X_val, y_val = X[train_sequence_count:], y[train_sequence_count:]
print(f"Training sequences: {len(X_train)}; validation sequences: {len(X_val)}")

lstm_model = SimpleLSTM(input_size=1, hidden_size=32)
training_history = lstm_model.fit(
    X_train.reshape(-1, sequence_length, 1),
    y_train,
    X_val.reshape(-1, sequence_length, 1),
    y_val,
    epochs=100,
    learning_rate=0.01,
    patience=12,
)

validation_predictions_scaled = np.array([
    lstm_model.forward(sequence.reshape(-1, 1)) for sequence in X_val
])
validation_predictions = scaler.inverse_transform(
    validation_predictions_scaled.reshape(-1, 1)
).ravel()
validation_targets = scaler.inverse_transform(y_val.reshape(-1, 1)).ravel()
validation_mae = mean_absolute_error(validation_targets, validation_predictions)
validation_rmse = np.sqrt(mean_squared_error(validation_targets, validation_predictions))
print(f"Best monitored validation loss (scaled MSE): {min(training_history['validation_loss']):.6f}")
print(f"Validation MAE: {validation_mae:.2f} units")
print(f"Validation RMSE: {validation_rmse:.2f} units")

In [ ]:
def generate_enhanced_wma_forecast(input_sequence: List[float]) -> float:
    """Forecast with normalized exponentially decreasing weights on recent demand."""
    if len(input_sequence) == 0:
        raise ValueError("input_sequence must contain at least one observation.")
    window_size = min(4, len(input_sequence))
    recent_data = np.asarray(input_sequence[-window_size:], dtype=float)
    weights = np.array([
        2.0 ** (-(window_size - 1 - i)) for i in range(window_size)
    ])
    weights /= weights.sum()
    return float(weights @ recent_data)


recent_data = extended_data[-sequence_length:]
recent_scaled = scaler.transform(np.asarray(recent_data).reshape(-1, 1)).ravel()
lstm_forecast_scaled = lstm_model.forward(recent_scaled.reshape(-1, 1))
lstm_forecast = float(
    scaler.inverse_transform([[lstm_forecast_scaled]])[0, 0]
)
wma_forecast = generate_enhanced_wma_forecast(recent_data)

# This fixed blend illustrates the hybrid equation; its weight is not learned here.
lstm_weight = 0.7
wma_weight = 1.0 - lstm_weight
hybrid_forecast = lstm_weight * lstm_forecast + wma_weight * wma_forecast

agreement = 1.0 - abs(lstm_forecast - wma_forecast) / max(
    abs((lstm_forecast + wma_forecast) / 2.0), 1e-12
)
agreement = float(np.clip(agreement, 0.0, 1.0))
if agreement > 0.85:
    confidence_label = "High agreement"
elif agreement > 0.75:
    confidence_label = "Moderate agreement"
else:
    confidence_label = "Low agreement"

print("=== One-step-ahead forecasts ===")
print(f"LSTM forecast: {lstm_forecast:.1f} units")
print(f"Enhanced WMA forecast: {wma_forecast:.1f} units")
print(f"Hybrid forecast: {hybrid_forecast:.1f} units")
print(f"Fixed ensemble weights: LSTM={lstm_weight:.1f}, WMA={wma_weight:.1f}")
print(f"Component agreement diagnostic: {agreement:.1%} ({confidence_label})")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("LSTM-Enhanced Hybrid Demand Forecast", fontsize=16, fontweight="bold")

# Show observed and synthetic periods separately so their different status is visible.
ax1 = axes[0, 0]
periods = np.arange(1, len(extended_data) + 1)
observed_periods = np.arange(1, len(original_demand) + 1)
ax1.plot(periods, extended_data, "o-", label="Generated series", alpha=0.7)
ax1.plot(observed_periods, original_demand, "o-", linewidth=2.5, label="Observed demand")
forecast_period = len(extended_data) + 1
ax1.scatter(forecast_period, hybrid_forecast, marker="D", s=85, label="Hybrid forecast")
ax1.scatter(forecast_period, lstm_forecast, marker="s", s=70, label="LSTM forecast")
ax1.scatter(forecast_period, wma_forecast, marker="^", s=70, label="WMA forecast")
ax1.set_xlabel("Period")
ax1.set_ylabel("Demand (units)")
ax1.set_title("Series and next-period forecasts")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
ax2.plot(periods, extended_data, "o-", label="Demand (units)")
ax2.plot(periods, scaled_data, "s-", label="Scaled demand")
ax2.axvline(train_end, color="black", linestyle="--", label="Train/validation boundary")
ax2.set_xlabel("Period")
ax2.set_ylabel("Value (units or scaled units)")
ax2.set_title("Training scale and chronological split")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
ax3.plot(training_history["train_loss"], label="Training MSE")
ax3.plot(training_history["validation_loss"], label="Validation MSE")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Mean squared error (scaled units)")
ax3.set_title("Training and validation loss")
ax3.legend()
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
methods = ["LSTM", "Enhanced WMA", "Hybrid"]
forecasts = [lstm_forecast, wma_forecast, hybrid_forecast]
bars = ax4.bar(methods, forecasts)
ax4.set_ylabel("Forecast (units)")
ax4.set_title("One-step forecast components")
ax4.grid(True, axis="y", alpha=0.3)
for bar, forecast in zip(bars, forecasts):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{forecast:.1f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

print("Plots show the generated example, model fitting diagnostics, and forecast components.")

## Results and interpretation

The code reports validation MAE and RMSE in demand units after converting predictions back from the scaled axis. MAE is the average absolute forecast error; RMSE is the square root of the average squared error and therefore gives larger errors greater influence. The loss plot shows the training objective and validation objective by epoch. Early stopping restores the parameter values associated with the lowest monitored validation loss.

The final forecast is produced for the period immediately following the generated series. The LSTM component reflects a fitted recurrent model; the WMA component reflects a transparent recency-weighted average. The hybrid forecast lies between the two component values when the weights are nonnegative and sum to one. Agreement between components is reported only as a heuristic diagnostic: it is not a measure of statistical uncertainty, forecast coverage, or calibrated confidence.

### What this example does and does not establish

- The synthetic extension makes it possible to demonstrate sequence construction, scaling, recurrent learning, chronological validation, and forecast blending with a short input series.
- Synthetic observations are generated from an assumed rule. They do not add independent empirical evidence, and validation performance on them does not establish performance on real demand.
- The ensemble share is fixed at 0.7 for the LSTM and 0.3 for the WMA. No ensemble-weight optimization is performed, so the weights should not be described as learned or optimal.
- The implemented LSTM has one recurrent layer and a linear output. Dropout, probabilistic forecasts, confidence intervals, multi-step forecasting, and automated hyperparameter search are not implemented.
- A practical deployment should evaluate forecasts using adequate real historical data and a validation design that reflects the intended forecasting horizon and decision process.

### Strengths and limitations of the method

**Potential strengths:** An LSTM can represent nonlinear relationships across the input sequence, while the WMA provides a simple, interpretable component. Blending can combine their forecasts according to a chosen weight.

**Limitations:** Neural-network training requires data and careful validation; results can depend on initialization and training choices. A small synthetic example cannot demonstrate superiority over other forecasting methods. The hybrid's behavior depends on the manually specified weight, and the agreement diagnostic is not uncertainty quantification.

### Natural extensions

The same forecasting setup can be evaluated on larger real datasets, additional input features, and longer forecast horizons. Further work can also investigate validation-based ensemble weights and formal uncertainty estimates. These extensions require additional modeling and evaluation and are not implemented in this notebook.